# Treinamento do Modelo

## Importando bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as sk
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from scipy import stats

# Funções

In [ ]:
def removeAtributoscomMesmoValor(dataset):

    dataset.drop(columns=dataset.columns[dataset.nunique()==1], inplace=True)

    return dataset

In [ ]:
def removeOutliers(dataset):

    # detect and remove outliers
    z_scores = stats.zscore(dataset)
    abs_z_scores = np.abs(z_scores)
    filtered_entries = (abs_z_scores < 5).all(axis=1)
    dataset = dataset[filtered_entries]
    # reset index
    #dataset = dataset.reset_index(drop=True)

    return dataset

In [ ]:
def removeDadosFaltantes(dataset):

    dataset=dataset.dropna()
    return dataset

In [ ]:
def normalization(X):

    scaler = sk.preprocessing.StandardScaler().fit(X)
    X = scaler.transform(X)
    return X

In [ ]:
def test_model(model, X, y, plot_matrix=False, img_name="confusion_matrix"):
  y_pred = model.predict(X)

  accuracy = accuracy_score(y, y_pred)
  precision = precision_score(y, y_pred, average="weighted")
  recall = recall_score(y, y_pred, average="weighted")
  f1 = f1_score(y, y_pred, average="weighted")

  cm = confusion_matrix(y, y_pred)

  # Display metrics
  print("\n" + "="*50)
  print("MODEL PERFORMANCE METRICS")
  print("="*50)
  print(f"Accuracy:  {accuracy:.4f}")
  print(f"Precision: {precision:.4f}")
  print(f"Recall:    {recall:.4f}")
  print(f"F1-Score:  {f1:.4f}")

  print("\n" + "="*50)
  print("DETAILED CLASSIFICATION REPORT")
  print("="*50)
  print(classification_report(y, y_pred))

  print("\n" + "="*50)
  print("CONFUSION MATRIX")
  print("="*50)
  print(cm)

  # Feature importance
  feature_importance = pd.DataFrame({
      'feature': X.columns,
      'importance': model.feature_importances_
  }).sort_values('importance', ascending=False)

  print("\n" + "="*50)
  print("FEATURE IMPORTANCE")
  print("="*50)
  print(feature_importance)

  if plot_matrix:
    # Plot confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=np.unique(y),
                yticklabels=np.unique(y))
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{img_name}.png", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def import_csv(file, show_info=False):
  data = pd.read_csv(file)

  # data.dropna(subset=["flags"], inplace=True)

  # data = removeAtributoscomMesmoValor(data)
  data = removeDadosFaltantes(data)
  # data["IPI"] = removeOutliers(data["IPI"])

  if show_info:
    print("Info:")
    data.info()
    print("\nDescrição estatística:")
    print(data.describe())
    print("\nValores nulos:")
    print(data.isnull().sum())
    print("\nDistribuição da target:")
    print(data['classification'].value_counts())
    # print(data["Flags"].value_counts())

  return data

## Importação dos dados

In [ ]:
data = import_csv(input_dataset, True)

In [ ]:
target = "classification"

X = data[features]
y = data[target]

# Treino do modelo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
)

model = RandomForestClassifier(random_state=42, **model_parameters)
model.fit(X_train, y_train)

## Teste do modelo

### Com base de treinos

In [ ]:
test_model(model, X_train, y_train, True, "confusion_matrix_train_data")

### Com base de Teste

In [ ]:
test_model(model, X_test, y_test, True, "confusion_matrix_test_data")

# Exportar Modelo

In [ ]:
import pickle

with open(f"{output_dir}/random_forest.pkl", "wb") as f:
  pickle.dump(model, f)